# Week 2 Day 3 Lab - Rejecting a payout

*Arthur's version of the assignment: a handler can approve or reject a payout, the agent looks again after a rejection, and a claim paused on one computer can be reviewed from another.*

This builds on [the lab](../../labs/week2day3.ipynb), so run that first. The graph is in [claim_graph.py](claim_graph.py), so that this notebook and the handler's script, [approve_claim.py](approve_claim.py), build exactly the same graph. The tables go in a schema of their own, `lab_reject`, away from the lab's.

## 3 parts:

PART 1: A handler who can say no

PART 2: CLM-1011, the best man's suitcase

PART 3: Approving from another computer

## Part 1: A handler who can say no

What changed from the lab's graph:

- **The handler's answer** carries `handler`, `approved` and `note`. Only `approved: True` approves, so a malformed answer can never pay a claim.
- **The approval node routes by itself.** It returns a `Command` that goes to `pay` on an approval. On a rejection, it adds the note to the conversation as a message from the handler and goes back to `agent`, which looks at the claim again.
- **After a rejection, every decision goes back to a handler.** Without this, an agent could lower the payout to under $1,000 and be paid without the handler who rejected it seeing the new figure.
- **Two rejections and the claim is referred**, so the agent and the handler can't go round forever.
- **Extra: the calculation goes with the pause.** The handler sees the payout tool's working, not just the total.
- **Extra: an audit trail.** Every review is kept in the state, and the `payments` table records how many rounds a claim took.

In [ ]:
import inspect
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import claim_graph
from claim_graph import MAX_REJECTIONS, build_graph, connect, review, start_claim, start_fresh, waiting_for_review

connection = connect()
graph = build_graph(connection)
display(Image(graph.get_graph().draw_mermaid_png()))

The dotted lines from `approval` are the `Command`'s possible destinations, declared with `destinations=` when the node is added. Here is the node, and the edge after `decide`.

In [ ]:
print(inspect.getsource(claim_graph.approval))
print(inspect.getsource(claim_graph.after_decide))

## Part 2: CLM-1011, the best man's suitcase

The handler paid $1,150. The catch is that the claim's lines aren't the policy's items. The jacket and the trousers are one tuxedo, and the camera body and its lens are one camera, so each pair counts once at the $300 per-item limit. The four shirts are on one line but are four items at $85, so they're paid in full.

In [ ]:
CLAIM = "CLM-1011"
config = {"configurable": {"thread_id": CLAIM}}
label = {decision["claim_id"]: decision for decision in json.loads(Path("../../data/evals/decisions.json").read_text(encoding="utf-8"))}[CLAIM]

print(label["reason"])
pd.DataFrame(label["calculation"])

The agent decides the claim and pauses for a handler. The pause carries the calculation behind the payout.

In [ ]:
start_fresh(connection, graph, CLAIM)
result = start_claim(graph, CLAIM)

waiting = result["__interrupt__"][0].value
print(f"{waiting['decision']} ${waiting['payout']:,.2f}: {waiting['rationale']}")
pd.DataFrame(waiting["calculation"])

### The handler reviews it

Dana approves a payout within a dollar of $1,150, and otherwise rejects it with a note that says what's wrong, as a handler would. `handle` resumes the thread with each answer until the claim is paid or referred.

In [ ]:
HANDLER = "Dana Whitfield"
NOTE = (
    "The jacket and trousers were sold as one two piece tuxedo, so they are one item. "
    "The camera body and its lens are one item too. "
    "The four dress shirts are four separate items at $85 each, not one item."
)


def handle(graph, result):
    """Review a paused claim as Dana would, until it's paid or referred. Returns the final result and the first payout."""
    first_payout = None
    while result.get("__interrupt__"):
        waiting = result["__interrupt__"][0].value
        first_payout = waiting["payout"] if first_payout is None else first_payout
        approved = waiting["decision"] == "approve" and abs(waiting["payout"] - label["payout"]) <= 1
        print(f"Round {waiting['round']}: {waiting['decision']} ${waiting['payout']:,.2f}, {'approved' if approved else 'rejected'}")
        result = review(graph, CLAIM, HANDLER, approved, "" if approved else NOTE)
    return result, first_payout


result, _ = handle(graph, result)
print(f"Finished: {result['decision']} ${result['payout']:,.2f}")

### An agent that gets it wrong

With Luna at `low`, the agent got $1,150 in 5 tries out of 5 while I built this, so the handler never needed to reject it. With reasoning turned off, as on Day 1, it paid $1,370 in 2 tries out of 5. So we review an agent with reasoning off, 5 times, and see whether the handler's note puts it right.

In [ ]:
quick_graph = build_graph(connection, reasoning_effort="none")

rows, reviewed = [], None
for attempt in range(1, 6):
    start_fresh(connection, quick_graph, CLAIM)
    result, first_payout = handle(quick_graph, start_claim(quick_graph, CLAIM))
    state = quick_graph.get_state(config)
    rows.append({"attempt": attempt, "first_payout": first_payout, "rounds": len(state.values["reviews"]), "final": result["decision"], "payout": result["payout"]})
    if len(state.values["reviews"]) > 1:
        reviewed = state

pd.DataFrame(rows)

### The audit trail

For the last attempt that needed more than one round, every review is in the state, with the decision the handler saw.

In [ ]:
pd.DataFrame(reviewed.values["reviews"]) if reviewed else "No attempt needed a second round this time"

The agent got the handler's note as a message in its conversation. These are the messages from the first rejection on: the note, the agent's second look, its new calculation and its answer.

In [ ]:
if reviewed:
    messages = reviewed.values["messages"]
    first_rejection = next(index for index, message in enumerate(messages) if message.type == "human" and "rejected your decision" in message.text)
    for message in messages[first_rejection:]:
        message.pretty_print()

### When the agent can't get it right

A handler who rejects without a useful note gives the agent nothing to work with. After `MAX_REJECTIONS` rejections, the claim ends as a referral, for a handler to settle by hand.

In [ ]:
start_fresh(connection, graph, CLAIM)
result = start_claim(graph, CLAIM)

while result.get("__interrupt__"):
    waiting = result["__interrupt__"][0].value
    print(f"Round {waiting['round']}: {waiting['decision']} ${waiting['payout']:,.2f}, rejected")
    result = review(graph, CLAIM, HANDLER, False, "This doesn't look right to me.")

print(f"Finished: {result['decision']} ${result['payout']:,.2f}. {result['rationale']}")
print(f"MAX_REJECTIONS is {MAX_REJECTIONS}")

## Part 3: Approving from another computer

The thread's state is in Supabase, not in this notebook. Anything that builds the same graph with the same `DATABASE_URL` can carry the claim on.

We pause CLM-1003, the fractured wrist in hospital, which the handler approved at $3,750, and leave it waiting.

In [ ]:
start_fresh(connection, graph, "CLM-1003")
result = start_claim(graph, "CLM-1003")
print(f"Waiting: {[item['claim_id'] for item in waiting_for_review(connection, graph)]}")

`approve_claim.py` is the handler's side. It builds the graph from `claim_graph.py`, lists the claims waiting for a handler, and approves or rejects one.

**On a second computer:** clone your fork, run `uv sync`, and add the same `DATABASE_URL` and `OPENROUTER_API_KEY` to `.env`. A rejection runs the agent again, so the second computer needs the OpenRouter key as well as the database. Both computers need the same package versions to read the saved state, which `uv sync` takes care of. Then, in the `undercover` folder:

```
uv run community_contributions/arthur_jzl/approve_claim.py
uv run community_contributions/arthur_jzl/approve_claim.py CLM-1003 --approve --handler "Priya Raman"
```

The cells below run the same commands in a separate process on this computer. The process shares no memory with this notebook, so everything it knows about the claim comes from Supabase.

In [ ]:
!{sys.executable} approve_claim.py

In [ ]:
!{sys.executable} approve_claim.py CLM-1003 --approve --handler "Priya Raman"

Back in the notebook, the payment made by the other process is in Supabase, and the thread has finished.

In [ ]:
print(graph.get_state({"configurable": {"thread_id": "CLM-1003"}}).next)
pd.DataFrame(connection.execute("SELECT * FROM payments ORDER BY paid_at").fetchall())

### My findings

From the run above, with GPT-6 Luna through OpenRouter, on October 28, 2026. The whole notebook costs a few cents.

**Luna at `low` usually gets CLM-1011 right.** It paid $1,150 in 5 tries out of 5 while I built this, and in Part 2 it was approved in one round. But in the vague-rejection test it started at $1,370, so `low` isn't always right either.

**With reasoning off, the handler's note fixed half the wrong payouts.** The first payout was wrong in 4 of 5 attempts: $1,370 three times and $1,500 once. After one rejection with the note, 2 of those 4 came back at exactly $1,150 and were paid.

**The other 2 overcorrected to $895, and the note was partly to blame.** The agent followed the note and treated the tuxedo and the camera as one item each. But it entered the shirts as `amount: 85, quantity: 4`, and the payout tool reads `amount` as the line's total, so it paid $85 for all four shirts. The note said "four separate items at $85 each", and the agent copied that figure into the tool call. A handler's note goes into the agent's next tool call too, so it's safer to give figures as line totals ("the shirts' line is $340 for four items"). The tool could also reject a line whose amount is less than its quantity times the smallest believable price, but that would be guessing.

**Sending every decision after a rejection back to a handler mattered.** $895 is under the $1,000 approval limit. Without that rule, the corrected decision would have been paid straight away, and the customer would have been underpaid by $255 with no handler seeing the new figure. With it, Dana saw $895, rejected it, and the claim was referred.

**The cap on rejections worked.** A note that only said "This doesn't look right to me" gave the agent nothing to work with: it went from $1,370 to $1,500, and after the second rejection the claim was referred, not left going round.

**The audit trail shows what happened.** The state keeps each review with the decision the handler saw, and the `payments` table records how many rounds each paid claim took.

**The stretch:** `approve_claim.py`, running as a separate process with no memory shared with the notebook, found CLM-1003 waiting in Supabase, showed its calculation and approved it. The notebook then saw the payment and a finished thread. On a real second computer, it needs the repo, `uv sync`, and the same `DATABASE_URL` and `OPENROUTER_API_KEY` in `.env`. The OpenRouter key is needed because a rejection runs the agent again.
